In [1]:
import polars as pl
from pathlib import Path

In [2]:
DATA_GENERAL = Path("../data_general")
DATA_PERSONAL = Path("../data_personal/Spotify Extended Streaming History")
OUT_DATA = Path('../output') 

In [3]:
lazy_general_df = pl.scan_parquet(DATA_GENERAL / "spotify_audio_features_*.parquet")

In [4]:
general_df = (
    lazy_general_df
    .filter(pl.col('null_response') == 0)   
    .drop('null_response')
    .collect()
)

In [5]:
general_df = general_df.rename({ 'id' : 'spotify_track_uri' })

In [6]:
display(general_df.head())

spotify_track_uri,name,popularity,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2Pe9cbhOTvOUTDE4bl7zzl""","""I dreamt you died""",0,630506,4,6,0,87.683,0.279,0.391,-12.054,0.32,0.816,0.737,0.177,0.0299
"""0wP732NKm8XgXu78XLRWoR""","""It's Death""",0,97216,4,5,1,105.298,0.429,0.318,-11.685,0.0566,0.587,0.782,0.202,0.36
"""22L6EJdnjx8oIo7GiF9hLe""","""Preliminary""",0,75180,4,0,1,117.657,0.283,0.581,-9.42,0.0555,0.923,0.939,0.106,0.0362
"""3a519lgQ13JXNi0G73mwMT""","""Disparage""",0,149447,4,5,0,100.685,0.244,0.995,-0.69,0.125,0.78,0.799,0.132,0.0634
"""27yP7p2lxWYTtnldRN8Kzx""","""Cut Down""",0,120816,4,7,1,123.499,0.313,0.618,0.411,0.073,0.843,0.109,0.126,0.187


In [7]:
personal_data_frames = {}
for num in range (2022,2027) :
    personal_data_frames[num] = pl.read_json(
        DATA_PERSONAL / f'Streaming_History_Audio_{num}.json',
        infer_schema_length=None  
        )

In [91]:
song_uri = '3U0UXxBIfjUsJ8RtxoxUFn'

In [92]:
stats = general_df.filter(pl.col('spotify_track_uri') == song_uri)

In [93]:
display(stats)

spotify_track_uri,name,popularity,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""3U0UXxBIfjUsJ8RtxoxUFn""","""Monolith""",64,264656,4,11,0,142.013,0.508,0.952,-6.445,0.0405,0.000409,0.756,0.114,0.799


In [112]:
# Subjectife paramaters (value given in percentage according to reverens value,
#  like 0.1 is mean : from 90% to 110% of reverense value)
smart_filter = [
    (0.0001,500),
    (0.001, 50),
    (0.01, 5),
    (0.1, 0.5),
    (0.2, 0.25),
    (0.4, 0.15),
    (0.6, 0.1),
    (0.8, 0.07),
    (1, 0.05),
]

subj_params = ['danceability','energy','speechiness','acousticness','instrumentalness','liveness','valence']

cofs = {}

for param in subj_params:
    for val, cof in smart_filter:
        if stats[param][0] < val:
            cofs[param] = cof
            break

#cof_dance = 0.1
#cof_energy = 0.2 
#cof_speech = 0.2
#cof_acoustic = 0.2
#cof_instr = 0.2 
#cof_live = 0.2
#cof_valence = 0.2

# Technical paramaters (in absalute value)
cof_tempo = 5 #bbm
cof_loud = 2 #Db

#subj_features = [       
#    ( cofs, 'danceability' ),     
#    (cof_energy, 'energy'),    
#    (cof_speech,  'speechiness' ),   
#    (cof_acoustic,'acousticness') ,
#    (cof_instr, 'instrumentalness'),    
# #   (cof_live, 'liveness'),    
#    (cof_valence, 'valence'),     
#]

tech_features = [        
    (cof_loud, 'loudness' )    
]

In [113]:
print(cofs)

{'danceability': 0.1, 'energy': 0.05, 'speechiness': 0.5, 'acousticness': 50, 'instrumentalness': 0.07, 'liveness': 0.25, 'valence': 0.07}


In [114]:
conditions = []


In [115]:
# Accept subjective features

for param in subj_params:
    state = stats[param]
    cond = pl.col(param).is_between(
        state * (1 - cofs[param]),
        state * (1 + cofs[param])
    )
    conditions.append(cond)


In [116]:
# Accept techical features

for cof, state_key in tech_features:

    state = stats[state_key]
    cond = pl.col(state_key).is_between(
        state - cof,
        state + cof
    )
    conditions.append(cond)


In [118]:
if conditions:
    result = (
        general_df.lazy()
        .filter(conditions)
        .filter( (pl.col('key') == stats['key']) & (pl.col('time_signature') == stats['time_signature']) & (pl.col('mode') == stats['mode']))
        .filter( (pl.col('tempo').is_between(stats['tempo'] - cof_tempo, stats['tempo'] + cof_tempo) ) | (pl.col('tempo').is_between((stats['tempo'] - cof_tempo)/2, (stats['tempo'] + cof_tempo) / 2) ) | (pl.col('tempo').is_between((stats['tempo'] - cof_tempo)*2, (stats['tempo'] + cof_tempo)*2) ))
        .collect()
    )

In [123]:
print(len(result))
result = result.with_columns(
    pl.format("spotify:track:{}", pl.col("spotify_track_uri")).alias("spotify_track_uri")
)
display(result.head())


20


spotify_track_uri,name,popularity,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""spotify:track:spotify:track:sp…","""慟哭""",0,251227,4,11,0,139.981,0.547,0.998,-4.668,0.0538,0.000048,0.769,0.0873,0.819
"""spotify:track:spotify:track:sp…","""Waterfall - Original Mix""",0,478123,4,11,0,138.994,0.516,0.955,-7.287,0.0524,0.0000337,0.762,0.0904,0.818
"""spotify:track:spotify:track:sp…","""The Message - Stoned Sun Remix""",0,586500,4,11,0,140.015,0.524,0.944,-4.729,0.0432,0.0000209,0.788,0.127,0.787
"""spotify:track:spotify:track:sp…","""GR3Y""",36,265159,4,11,0,145.007,0.517,0.929,-5.299,0.0378,0.000005,0.747,0.0939,0.76
"""spotify:track:spotify:track:sp…","""Annihilation""",1,242341,4,11,0,144.985,0.552,0.928,-6.152,0.0358,0.000129,0.793,0.0859,0.788


In [120]:
file_path = OUT_DATA / f"Like '{stats['name'][0]}'.csv"
result.select('spotify_track_uri').write_csv(file_path , separator=',')